# Verifying multi-telescope occultation candidates

Cross-matches per-telescope detections by **RA/Dec + time**, then **vets every 2+ telescope coincidence** for being a real astrophysical occultation (tier logic + per-dip metrics), flagging which need follow-up. Drill into any candidate to see per-telescope status & timing, the overlaid light curves, an on-demand reconstruction of a missing telescope (across minute boundaries, solving a WCS if needed), and the event-image grid.

**Flow:** config → load detections → RA/Dec matching → candidate summary → *(helper defs)* → **batch vetting** → **drill into a candidate**.

In [ ]:
import os
import sys

# Path to the ColibriPipeline package so we can reuse the production helpers.
# Reuse only path-agnostic library modules (cir/cp/getRAdec/astrometrynet_funcs)
# -- NOT generate_specific_lightcurve, which hard-codes COMPUTERNAME and D:/ paths.
path_add = r"C:\Users\GreenBird\Documents\GitHub\ColibriPipeline\ColibriPipeline"
if path_add not in sys.path:
    sys.path.append(path_add)

import glob
import warnings
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sep
from astropy import wcs
from astropy.io import fits
from astropy.time import Time

import colibri_image_reader as cir
import colibri_photometry as cp
import getRAdec
import astrometrynet_funcs

# Pipeline-wide constants (mirrors generate_specific_lightcurve.py / lightcurve_maker.py)
MINDIR_FORMAT    = "%Y%m%d_%H.%M.%S.%f"   # minute-directory name format
TIMESTAMP_FORMAT = "%Y-%m-%dT%H:%M:%S.%f"  # DATE-OBS header format
EXPOSURE_TIME    = 0.025   # s/frame
APERTURE_RADIUS  = 3.0     # px
EXCLUDE_IMAGES   = 1
SEC_TO_SAVE      = 1.0     # s on either side of the event
DRIFT_TOLERANCE  = 1.0     # px/s: above this we refuse to build a curve
DRIFT_THRESHOLD  = 0.025   # px/s: above this we apply a drift correction
COORD_TOLERANCE  = 0.002   # deg (~7"): RA/Dec match tolerance (simultaneous_occults.py)
TIME_TOLERANCE   = 1.0     # s: event-time match tolerance
IMG_DIM          = 2048    # detector size (px), used for FOV bounds checks
NUM_TO_SKIP      = 1       # frames to skip when median-stacking for WCS (colibri_main_py3.py)
NUM_TO_STACK     = 9       # frames to median-stack for WCS
WCS_ORDER        = 4       # SIP polynomial order for astrometry solve

In [ ]:
@dataclass
class Telescope:
    """Everything path-dependent for one telescope, so no ad-hoc drive-letter
    rewrites are scattered through the notebook."""
    name:    str          # "RED" / "GREEN" / "BLUE" (matches *BIRD suffix)
    color:   str          # plot colour
    archive: str          # ColibriArchive root (det/art/.npy/.fits live here)
    data:    str          # ColibriData root (raw minute dirs + .rcd live here)
    drive:   str          # drive letter, e.g. "R:"  (for rewriting stored .rcd paths)
    det_files: list = field(default_factory=list)  # populated below

    @property
    def archive_date(self):
        return os.path.join(self.archive, OBS_DATE)            # hyphenated date

    @property
    def data_date(self):
        return os.path.join(self.data, OBS_DATE.replace("-", ""))  # YYYYMMDD


# ------------------------------------------------------------------
# Configure the night and the telescopes to consider.
# ------------------------------------------------------------------
OBS_DATE = "2026-06-16"   # hyphenated, as used under ColibriArchive

TELESCOPES = [
    Telescope("RED",   "red",   archive=r"R:\ColibriArchive", data=r"R:\ColibriData", drive="R:"),
    Telescope("GREEN", "green", archive=r"D:\ColibriArchive", data=r"D:\ColibriData", drive="D:"),
    Telescope("BLUE",  "blue",  archive=r"B:\ColibriArchive", data=r"B:\ColibriData", drive="B:"),
]
TEL_BY_NAME = {t.name: t for t in TELESCOPES}


def load_det_files(tel):
    """Detection .txt files in a telescope's archive for OBS_DATE.

    Missing archive (e.g. telescope offline / drive not mounted) is reported,
    not raised, so a telescope can legitimately have zero detections."""
    d = tel.archive_date
    if not os.path.isdir(d):
        print(f"WARNING: {tel.name} archive not found: {d}  (telescope offline / drive unmounted?)")
        return []
    return [f for f in os.listdir(d) if f.startswith("det") and f.endswith(".txt")]


for tel in TELESCOPES:
    tel.det_files = load_det_files(tel)
    print(f"{tel.name:5s}: {len(tel.det_files)} detection files in {tel.archive_date}")

# Quick peek
for tel in TELESCOPES:
    if tel.det_files:
        print(f"  e.g. {tel.det_files[0]}")
        break

In [ ]:
def parse_header(file_path):
    """Parse the commented '# key: value' header of a det_/art_ .txt file."""
    metadata = {}
    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            if line.startswith("#"):
                line_content = line[1:].strip()
                if ":" in line_content:
                    key, value = map(str.strip, line_content.split(":", 1))
                    metadata[key] = value
            else:
                break  # stop at the data table
    return metadata


# ------------------------------------------------------------------
# Header / filename accessors
# ------------------------------------------------------------------
def get_event_datetime(filename):
    """Full event datetime parsed from a det_/art_ filename.

    det_2026-06-16_050404_531339000_starN_TELESCOPE.txt
        -> date=2026-06-16, time=050404 (HHMMSS), frac=531339000 (ns).
    Parsing the *date* too (not just the time) avoids spurious matches across
    midnight."""
    parts = os.path.basename(filename).split("_")
    date_str = parts[1]                       # YYYY-MM-DD
    hms      = parts[2]                        # HHMMSS
    frac_us  = parts[3][:6]                    # ns -> us (6 digits)
    return datetime.strptime(f"{date_str}_{hms}.{frac_us}", "%Y-%m-%d_%H%M%S.%f")


def get_radec(archive_dir, filename):
    """(RA, Dec) in degrees from the 'RA Dec Coords' header line.

    Returns (inf, inf) and warns if the line is missing -- this means the file
    predates WCS solving (see coordsfinder.py) and the data needs reprocessing
    before it can be cross-matched by sky position."""
    meta = parse_header(os.path.join(archive_dir, filename))
    if "RA Dec Coords" not in meta:
        warnings.warn(f"No 'RA Dec Coords' in {filename} -- reprocess (no WCS solution).")
        return float("inf"), float("inf")
    ra, dec = (float(v) for v in meta["RA Dec Coords"].split()[:2])
    return ra, dec


def get_star_coords(archive_dir, filename):
    """Pixel (x, y) of the star on that telescope, from the header."""
    meta = parse_header(os.path.join(archive_dir, filename))
    return np.array([float(c) for c in meta["Star Coords"].split()])


def radec_match(c1, c2, tol_deg=COORD_TOLERANCE):
    """Declination-corrected angular separation test (simultaneous_occults.py:373)."""
    if not (np.all(np.isfinite(c1)) and np.all(np.isfinite(c2))):
        return False
    ra1, dec1 = c1
    ra2, dec2 = c2
    sep_deg = np.hypot((ra1 - ra2) / np.cos(np.radians(dec1)), dec1 - dec2)
    return sep_deg <= tol_deg


# ------------------------------------------------------------------
# Path / minute-directory helpers
# ------------------------------------------------------------------
def rcd_path(tel, raw_path):
    """Rewrite a stored .rcd path's drive letter to telescope `tel`'s drive.

    Detection files record .rcd paths with whatever drive the *processing*
    machine used; to read them from another telescope's archive we swap the
    leading 'X:' prefix.  Replaces the scattered '.replace(\"d:\", \"B:\")' hacks."""
    raw_path = str(raw_path)
    if len(raw_path) >= 2 and raw_path[1] == ":":
        return tel.drive + raw_path[2:]
    return raw_path


def list_minute_dirs(tel):
    """[(start_datetime, name, fullpath)] for every minute dir of OBS_DATE.

    Existence of a covering minute directory is the ground truth for "was this
    telescope observing then?"."""
    out = []
    d = tel.data_date
    if not os.path.isdir(d):
        return out
    for name in os.listdir(d):
        full = os.path.join(d, name)
        if not os.path.isdir(full) or name == "Dark":
            continue
        try:
            t = datetime.strptime(name + "000", MINDIR_FORMAT)  # name has ms; pad to us
        except ValueError:
            continue
        out.append((t, name, full))
    return sorted(out)


def find_minute_dir(tel, event_dt):
    """Minute dir whose [t, t+60s) window contains event_dt, else None."""
    for t, name, full in list_minute_dirs(tel):
        if t <= event_dt < t + timedelta(minutes=1.1):
            return name, full
    return None

In [ ]:
# ==================================================================
# Cross-telescope matching by TIME + RA/Dec (not raw pixels!)
# ------------------------------------------------------------------
# Telescopes have different fields/orientations, so the same star sits at
# different pixel coords on each one.  We match on sky position (RA/Dec from
# the header, written by coordsfinder.py) within COORD_TOLERANCE, exactly like
# the production simultaneous_occults.py.
#
# Crucially, PARTIAL matches are kept: a candidate seen by 2 of 3 telescopes is
# recorded with the third marked missing -- it is NOT discarded.  That is the
# whole point of investigating a "Red is missing" event.
# ==================================================================

# Pre-compute (event time, RA/Dec) for every detection, once.
records = {}
for tel in TELESCOPES:
    recs = []
    for f in tel.det_files:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")  # summarised below instead of per-file
            radec = np.array(get_radec(tel.archive_date, f))
        recs.append({"file": f, "dt": get_event_datetime(f), "radec": radec})
    records[tel.name] = recs
    n_nowcs = sum(1 for r in recs if not np.all(np.isfinite(r["radec"])))
    if n_nowcs:
        print(f"WARNING: {tel.name}: {n_nowcs}/{len(recs)} detections lack RA/Dec "
              f"(no WCS) and cannot be sky-matched -- reprocess to fix.")


def build_candidates():
    """Group detections into coincident candidates across telescopes.

    Returns a list of dicts: {files: {tel: filename}, radec, event_dt,
    present: [tel...], missing: [tel...]}.  Only multi-telescope coincidences
    (>=2) are returned; isolated single-telescope detections are ignored."""
    consumed = set()          # (tel_name, filename) already assigned to a candidate
    candidates = []
    for anchor in TELESCOPES:
        for arec in records[anchor.name]:
            key = (anchor.name, arec["file"])
            if key in consumed:
                continue
            matched = {anchor.name: arec["file"]}
            consumed.add(key)
            for other in TELESCOPES:
                if other.name == anchor.name:
                    continue
                for orec in records[other.name]:
                    okey = (other.name, orec["file"])
                    if okey in consumed:
                        continue
                    if (abs((arec["dt"] - orec["dt"]).total_seconds()) <= TIME_TOLERANCE
                            and radec_match(arec["radec"], orec["radec"])):
                        matched[other.name] = orec["file"]
                        consumed.add(okey)
                        break
            present = [t.name for t in TELESCOPES if t.name in matched]
            if len(present) < 2:
                continue  # not a coincidence
            candidates.append({
                "files":    matched,
                "radec":    arec["radec"],
                "event_dt": arec["dt"],
                "present":  present,
                "missing":  [t.name for t in TELESCOPES if t.name not in matched],
            })
    return candidates


candidates = build_candidates()

for tel in TELESCOPES:
    print(f"Number of {tel.name:5s} files: {len(tel.det_files)}")
print(f"\nNumber of coincident candidates: {len(candidates)}")
n_partial = sum(1 for c in candidates if c["missing"])
print(f"  of which partial (a telescope missing): {n_partial}")

In [ ]:
# Summarise candidates.  Partial candidates (a telescope missing) are flagged
# up front -- these are the ones to investigate.
summary_rows = []
for i, c in enumerate(candidates):
    summary_rows.append({
        "idx":      i,
        "event":    c["event_dt"].strftime("%H:%M:%S.%f")[:-3],
        "RA":       round(c["radec"][0], 4),
        "Dec":      round(c["radec"][1], 4),
        "present":  ",".join(c["present"]),
        "MISSING":  ",".join(c["missing"]) or "-",
    })
candidates_df = pd.DataFrame(summary_rows)
print(candidates_df.to_string(index=False) if len(candidates_df) else "No candidates.")

# Indices of partial candidates, for quick selection below.
partial_idxs = [r["idx"] for r in summary_rows if r["MISSING"] != "-"]
print(f"\nPartial-candidate indices (a telescope missing): {partial_idxs}")

In [ ]:
# ------------------------------------------------------------------
# Drill-in plot helpers (definitions only; called from the drill-in section).
# ------------------------------------------------------------------
def load_lightcurve(tel, filename):
    """(DataFrame, header dict) for one det_/art_ file."""
    path = os.path.join(tel.archive_date, filename)
    df = pd.read_csv(path, delim_whitespace=True, skiprows=lambda x: x < 19,
                     names=["filename", "time", "flux", "conv_flux"])
    return df, parse_header(path)


def plot_candidate(idx, normalize=True):
    """Overlay the light curves of whichever telescopes are present."""
    cand = candidates[idx]
    plt.figure(figsize=(10, 5))
    for name in cand["present"]:
        tel = TEL_BY_NAME[name]
        df, meta = load_lightcurve(tel, cand["files"][name])
        sig = meta.get("significance", "n/a")
        flux = df["flux"] / np.mean(df["flux"]) if normalize else df["flux"]
        plt.plot(df["time"], flux, marker="o", markersize=2,
                 color=tel.color, label=f"{name} (sigma={sig})")
    miss = ", ".join(cand["missing"]) or "none"
    plt.xlabel("Time (s)")
    plt.ylabel("Flux / mean" if normalize else "Flux")
    plt.title(f"Candidate {idx} @ {cand['event_dt'].strftime('%H:%M:%S')}  "
              f"RA={cand['radec'][0]:.4f} Dec={cand['radec'][1]:.4f}   MISSING: {miss}")
    plt.legend(); plt.grid()
    plt.show()


def show_first_frames(idx):
    """Zoomed first-frame image per present telescope, with the star circled
    (reads each telescope's .rcd from its own drive via rcd_path)."""
    cand = candidates[idx]
    present = cand["present"]
    fig, axes = plt.subplots(1, len(present), figsize=(12 * len(present), 10), squeeze=False)
    for ax, name in zip(axes[0], present):
        tel = TEL_BY_NAME[name]
        df, _ = load_lightcurve(tel, cand["files"][name])
        frame_paths = [rcd_path(tel, p) for p in df["filename"].values]
        images = cir.importFramesRCD(frame_paths, 30, 10)[0]
        sx, sy = get_star_coords(tel.archive_date, cand["files"][name])
        ax.imshow(np.log(images[0]), cmap="gray")
        ax.add_patch(plt.Circle((sx, sy), 10, color="red", fill=False, linewidth=2))
        ax.set_xlim(sx - 50, sx + 50); ax.set_ylim(sy - 50, sy + 50)
        ax.set_title(f"{name} first frame (star at {sx:.0f},{sy:.0f})")
        ax.set_xticks([]); ax.set_yticks([])
    plt.show()

# Verification helpers

Definitions used by the batch vetting and drill-in sections below. For a given candidate, per telescope, cheapest check first:

1. **Observing?** — a minute directory under `ColibriData/{YYYYMMDD}` covering the event time. If not, that telescope simply **was not observing** this star.
2. **Processed?** — does `{minute}_stars.npy` exist.
3. **In FOV?** — does the star's RA/Dec land on the detector (solved star list, or reverse-mapped through the minute's `*_wcs.fits`).

`verify_candidate` also reports the **per-telescope frame-time offset** (from the `unix_time` column of `_stars.npy`) to expose a clock/timestamp mismatch.

In [ ]:
# ==================================================================
# Verify WHY a telescope is missing, and check for a timing offset.
# ------------------------------------------------------------------
# For each telescope we answer, cheapest check first:
#   observing? -> was there a minute dir covering the event time
#   processed? -> does {minute}_stars.npy exist
#   in FOV?    -> does the star (RA/Dec) land on the detector
# and we report the cross-telescope frame-time offset from _stars.npy.
# ==================================================================
def load_wcs(tel, minutedir_name):
    """astropy WCS for a minute's solved frame, or None if not solved.

    Reads the existing *_wcs.fits in the archive; does NOT trigger an
    astrometry.net solve (so it is safe/fast to call here)."""
    hits = glob.glob(os.path.join(tel.archive_date, f"*{minutedir_name}*wcs*.fits"))
    if not hits:
        return None
    with fits.open(hits[0]) as hdul:
        return wcs.WCS(hdul[0].header)


def check_in_fov(tel, minutedir_name, radec, tol=COORD_TOLERANCE):
    """Return (in_fov, (x, y)).  in_fov is True/False, or None if undeterminable
    (no WCS-tagged star list and no *_wcs.fits)."""
    # Prefer the solved star list (columns x, y, r, ra, dec) if present.
    for p in glob.glob(os.path.join(tel.archive_date, f"{minutedir_name}_*sig_pos.npy")):
        arr = np.load(p, allow_pickle=True)
        if getattr(arr, "ndim", 0) == 2 and arr.shape[1] >= 5:
            d = np.hypot((arr[:, 3] - radec[0]) / np.cos(np.radians(radec[1])),
                         arr[:, 4] - radec[1])
            j = int(np.argmin(d))
            if d[j] <= tol:
                return True, (float(arr[j, 0]), float(arr[j, 1]))
    # Otherwise reverse-map RA/Dec through the WCS and test detector bounds.
    transform = load_wcs(tel, minutedir_name)
    if transform is not None:
        px = getRAdec.getXYSingle(transform, (radec[0], radec[1]))
        return (0 <= px[0] < IMG_DIM and 0 <= px[1] < IMG_DIM), (float(px[0]), float(px[1]))
    return None, None


def load_frame_times(tel, minutedir_name):
    """Per-frame unix times (s) from {minute}_stars.npy, or None."""
    hits = glob.glob(os.path.join(tel.archive_date, f"{minutedir_name}_stars.npy"))
    if not hits:
        return None
    arr = np.load(hits[0], allow_pickle=True)   # (n_frames, n_stars, 4) = [x, y, flux, unix]
    return arr[:, 0, 3].astype(float)


def verify_candidate(idx):
    cand = candidates[idx]
    radec, ev = cand["radec"], cand["event_dt"]
    print(f"=== Candidate {idx} @ {ev}  RA={radec[0]:.4f} Dec={radec[1]:.4f} ===")
    rows, frame_times = [], {}
    for tel in TELESCOPES:
        name = tel.name
        has_det = name in cand["files"]
        md = find_minute_dir(tel, ev)
        observing = md is not None
        mdname = md[0] if md else None
        processed = bool(glob.glob(os.path.join(tel.archive_date, f"{mdname}_stars.npy"))) if mdname else False
        in_fov, px = check_in_fov(tel, mdname, radec) if mdname else (None, None)
        if processed:
            frame_times[name] = load_frame_times(tel, mdname)

        # Verdict, cheapest explanation first.
        if has_det:
            verdict = "DETECTED"
        elif not observing:
            verdict = "NOT OBSERVING (no minute dir covering event) -> did not observe this star"
        elif not processed:
            verdict = "observing but minute NOT processed (no _stars.npy)"
        elif in_fov is False:
            verdict = "star OUTSIDE FOV -> not observable here"
        elif in_fov is None:
            verdict = "no WCS solution -> cannot confirm FOV (reprocess/solve)"
        else:
            verdict = "observable (observing+in FOV) but NO det/art -> forced-curve job likely failed / sub-threshold"

        rows.append({"tel": name, "det": has_det, "observing": observing,
                     "minute_dir": mdname, "processed": processed,
                     "in_fov": in_fov,
                     "px": None if px is None else (round(px[0]), round(px[1])),
                     "verdict": verdict})
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))

    # Timing offset between telescopes (frame times from _stars.npy).
    avail = {k: v for k, v in frame_times.items() if v is not None and len(v)}
    if len(avail) >= 2:
        ref = next(iter(avail))
        print(f"\nFrame-time check (ref = {ref}):")
        for name, t in avail.items():
            tag = "  <-- ref" if name == ref else \
                  f"  median offset vs {ref}: {np.median(t) - np.median(avail[ref]):+.4f} s"
            print(f"  {name}: nframes={len(t)}  "
                  f"first={Time(t[0], format='unix').iso[11:]}  "
                  f"last={Time(t[-1], format='unix').iso[11:]}{tag}")
    else:
        print("\nFrame-time check: need _stars.npy from >=2 telescopes for this minute.")
    return df

## Reconstruct the missing telescope's light curve

Rebuild the missing telescope's light curve directly from the raw `.rcd` frames at the WCS-mapped pixel — the same photometry the pipeline uses for `art_` curves — so you can see the dip even when no `det_`/`art_` file was ever written. This handles two real-world wrinkles:

- **Event straddles a minute boundary.** If the telescope's minute directories are offset in time, the event window falls across two dirs. `relevant_minute_dirs` finds *every* dir the window touches, and frames are gathered (and time-filtered) from each, then stitched together.
- **No WCS solution.** If a minute dir has no `*_wcs.fits`, `ensure_wcs` builds the median stack (`cir.stackImages`) and solves it via `astrometrynet_funcs.getLocalSolution` (local WSL `solve-field`, falling back to the astrometry.net web API), persisting `high{minute}_wcs.fits` into the archive.

Everything is plotted on a **seconds-relative-to-event** axis (event at 0) so the reconstructed curve lines up with the telescopes that did detect — even where its seconds-within-minute wraps across the boundary.

In [ ]:
# ==================================================================
# Forced light-curve reconstruction for a telescope with no det_/art_ file.
# ------------------------------------------------------------------
# Handles the real-world case where the event straddles TWO minute directories
# (telescope minute boundaries are offset from one another): we gather the
# event's frames from every minute dir that overlaps the window, solve a WCS on
# demand if one is missing, do aperture photometry at the per-dir WCS-mapped
# pixel, and stitch everything onto an event-relative time axis.
# ==================================================================
def wsl_path(win_path):
    """Windows path 'X:\\a\\b' -> WSL mount '/mnt/x/a/b' (for solve-field)."""
    p = str(win_path)
    if len(p) >= 2 and p[1] == ":":
        return f"/mnt/{p[0].lower()}/" + p[2:].replace("\\", "/").lstrip("/")
    return p.replace("\\", "/")


def masterdark_array(tel):
    """[(datetime, path)] of this telescope's master darks (for cir.chooseDark)."""
    mdark_dir = os.path.join(tel.archive_date, "masterDarks")
    return np.array([(datetime.strptime(d[:-5] + "000", MINDIR_FORMAT),
                      os.path.join(mdark_dir, d)) for d in os.listdir(mdark_dir)])


def ensure_medstack(tel, minute_name, minute_path, dark):
    """Return the medstack path, building high{minute}_medstacked.fits if absent."""
    archive = Path(tel.archive_date)
    medstack = archive / f"high{minute_name}_medstacked.fits"
    if not medstack.exists():
        print(f"  {tel.name}/{minute_name}: building medstack ...")
        cir.stackImages(Path(minute_path), archive, NUM_TO_SKIP, NUM_TO_STACK, dark)
    return medstack


def ensure_wcs(tel, minute_name, minute_path, dark):
    """WCS for a minute dir; solve + persist high{minute}_wcs.fits if missing.

    Reuses the pipeline solver (local WSL solve-field, web API fallback)."""
    transform = load_wcs(tel, minute_name)
    if transform is not None:
        return transform
    print(f"  {tel.name}/{minute_name}: no WCS -- solving (this can take a while) ...")
    medstack = ensure_medstack(tel, minute_name, minute_path, dark)
    wcs_name = f"high{minute_name}_wcs.fits"
    header = astrometrynet_funcs.getLocalSolution(wsl_path(medstack), wcs_name, WCS_ORDER)
    if header is None:
        print(f"  {tel.name}/{minute_name}: WCS solve FAILED.")
        return None
    # Persist into the archive so load_wcs finds it next time.
    wcs_out = Path(tel.archive_date) / wcs_name
    try:
        header.tofile(wcs_out, overwrite=True)
    except TypeError:                      # older astropy: no overwrite kwarg
        if wcs_out.exists():
            wcs_out.unlink()
        header.tofile(wcs_out)
    return wcs.WCS(header)


def relevant_minute_dirs(tel, event_dt, sec_window):
    """All minute dirs whose time span overlaps [event +/- sec_window] (+buffer).

    Returns [(name, fullpath, start_dt)] -- usually the two dirs bracketing an
    event that lands near a minute boundary."""
    dirs = list_minute_dirs(tel)              # [(start, name, full)] sorted
    lo = event_dt - timedelta(seconds=sec_window + 2)
    hi = event_dt + timedelta(seconds=sec_window + 2)
    out = []
    for i, (start, name, full) in enumerate(dirs):
        nxt = dirs[i + 1][0] if i + 1 < len(dirs) else start + timedelta(seconds=65)
        end = min(nxt, start + timedelta(seconds=65))
        if start <= hi and end >= lo:
            out.append((name, full, start))
    return out


def reconstruct_lightcurve(tel, event_dt, radec, sec_window=SEC_TO_SAVE):
    """(t_rel_s, flux) for the star at `radec` on `tel`, built from raw .rcd
    frames across however many minute dirs the event window touches.

    t_rel is seconds relative to the event.  Returns None if nothing usable."""
    dirs = relevant_minute_dirs(tel, event_dt, sec_window)
    if not dirs:
        print(f"{tel.name}: no minute dir near {event_dt} -- not observing.")
        return None

    event_unix = Time(event_dt).unix
    mdarks = masterdark_array(tel)
    obs_dt = datetime.strptime(OBS_DATE, "%Y-%m-%d")
    margin = 40                               # extra frames each side before time-filtering
    times, flux = [], []

    print(f"{tel.name}: event spans {len(dirs)} minute dir(s): {[d[0] for d in dirs]}")
    for name, full, start in dirs:
        dark = cir.chooseDark(Path(full), mdarks, obs_dt)
        transform = ensure_wcs(tel, name, full, dark)
        if transform is None:
            print(f"  {name}: no WCS; skipping this dir.")
            continue
        sx, sy = (float(v) for v in getRAdec.getXYSingle(transform, (radec[0], radec[1])))
        if not (0 <= sx < IMG_DIM and 0 <= sy < IMG_DIM):
            print(f"  {name}: star at ({sx:.0f},{sy:.0f}) outside FOV; skipping this dir.")
            continue

        image_paths = sorted(Path(full).glob("*.rcd"))
        n = len(image_paths)
        est_c = (event_dt - start).total_seconds() / EXPOSURE_TIME
        f_lo = max(EXCLUDE_IMAGES + 1, int(est_c - sec_window / EXPOSURE_TIME - margin))
        f_hi = min(n, int(est_c + sec_window / EXPOSURE_TIME + margin) + 1)

        kept = 0
        for i in range(f_lo, f_hi):
            img, it = cir.importFramesRCD(image_paths, i, 1, dark)   # 2D frame, [time]
            t_unix = Time(it[0], precision=9).unix
            if abs(t_unix - event_unix) > sec_window:
                continue
            times.append(t_unix)
            flux.append(sep.sum_circle(img, [sx], [sy], [APERTURE_RADIUS])[0][0])
            kept += 1
        print(f"  {name}: kept {kept} frames within +/-{sec_window}s at px ({sx:.0f},{sy:.0f})")

    if not times:
        print(f"{tel.name}: NO frames within +/-{sec_window}s of {event_dt}.")
        return None
    order = np.argsort(times)
    return np.array(times)[order] - event_unix, np.array(flux)[order]


def overlay_reconstruction(idx, target_name, sec_window=SEC_TO_SAVE):
    """Plot present telescopes' curves and overlay the reconstructed (dashed)
    curve for `target_name`, all on a seconds-relative-to-event axis so a
    boundary-crossing curve lines up with the others."""
    cand = candidates[idx]
    tel = TEL_BY_NAME[target_name]
    res = reconstruct_lightcurve(tel, cand["event_dt"], cand["radec"], sec_window)

    plt.figure(figsize=(10, 5))
    for name in cand["present"]:
        ptel = TEL_BY_NAME[name]
        f = cand["files"][name]
        df, meta = load_lightcurve(ptel, f)
        ev = get_event_datetime(f)                       # this telescope's own event time
        ev_sec = ev.second + ev.microsecond / 1e6        # seconds within its minute
        plt.plot(df["time"] - ev_sec, df["flux"] / np.mean(df["flux"]),
                 marker="o", markersize=2, color=ptel.color,
                 label=f"{name} (sigma={meta.get('significance','n/a')})")
    if res is not None:
        t, fl = res
        plt.plot(t, fl / np.mean(fl), linestyle="--", marker=".", markersize=3,
                 color=tel.color, label=f"{target_name} (reconstructed, {len(t)} frames)")
    plt.axvline(0, color="k", linestyle=":", alpha=0.5)
    plt.xlabel("Time relative to event (s)"); plt.ylabel("Flux / mean")
    plt.title(f"Candidate {idx}: reconstructed {target_name} overlaid")
    plt.legend(); plt.grid(); plt.show()


# Event-image grid (drill-in helper)

`plot_images_in_grid` / `show_event_images` render the frames around an event zoomed on the star — used in the drill-in section to confirm a dip is the star fading, not a streak or a cosmic ray on a neighbour.

In [ ]:
# add a circle to the image at the star coords
from matplotlib.patches import Circle

def plot_images_in_grid(images, times, star_coords, zoom_size=50):
    """
    Plots images in a 3x3 grid, zoomed around the star's coordinates, with times in the titles.
    vmin and vmax are based on the first zoomed image.
    
    Parameters:
        images (list of np.ndarray): List of image arrays.
        times (list of str): List of times corresponding to each image (as strings).
        star_coords (tuple): Coordinates of the star (x, y).
        zoom_size (int): Half the size of the zoomed region around the star.
    """
    # Extract the first zoomed image to determine vmin and vmax
    x, y = int(star_coords[0]), int(star_coords[1])
    first_zoomed_image = images[0][max(0, y-zoom_size):y+zoom_size, max(0, x-zoom_size):x+zoom_size]
    vmin, vmax = np.min(first_zoomed_image), np.max(first_zoomed_image)
    # alternative vmin and vmax of area 5x the zoom size around the star
    # vmin, vmax = np.min(images[0][max(0, y-5*zoom_size):y+5*zoom_size, max(0, x-5*zoom_size):x+5*zoom_size]), np.max(images[0][max(0, y-5*zoom_size):y+5*zoom_size, max(0, x-5*zoom_size):x+5*zoom_size])
    



    num_images = len(images)
    for i in range(0, num_images, 9):
        fig, axes = plt.subplots(3, 3, figsize=(20, 20))
        fig.suptitle(f"Images {i+1} to {min(i+9, num_images)}", fontsize=16)
        
        for j, ax in enumerate(axes.flat):
            if i + j < num_images:
                image = images[i + j]
                time = times[i + j]  # Use the string time directly
                
                # Extract zoomed region
                zoomed_image = image[max(0, y-zoom_size):y+zoom_size, max(0, x-zoom_size):x+zoom_size]
                
                # Plot the zoomed image without logarithmic scaling
                ax.imshow(zoomed_image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)


                # Add a circle around x-y
                circle = Circle(star_coords, 10, color='red', fill=False, linewidth=2)
                ax.add_patch(circle)
                ax.set_title(f"Image {i + j + 1}\nTime: {time}")
                ax.axis('off')
            else:
                ax.axis('off')  # Hide unused subplots
        
        plt.tight_layout()
        plt.subplots_adjust(top=0.9)  # Adjust for the title
        plt.show()

In [ ]:
# ------------------------------------------------------------------
# Drill-in: zoomed image grid around the star across the event frames, for one
# telescope of a candidate. Lets you confirm the dip is the star fading rather
# than a streak / cosmic ray on a neighbour.
# ------------------------------------------------------------------
def show_event_images(idx, tel_name, num=80, zoom=50):
    cand = candidates[idx]
    if tel_name not in cand["present"]:
        print(f"{tel_name} has no det file for candidate {idx}; "
              f"use overlay_reconstruction to rebuild it instead.")
        return
    tel = TEL_BY_NAME[tel_name]
    df, meta = load_lightcurve(tel, cand["files"][tel_name])
    frame_paths = [rcd_path(tel, p) for p in df["filename"].values]
    images, times = cir.importFramesRCD(frame_paths, 0, num)
    sx, sy = get_star_coords(tel.archive_date, cand["files"][tel_name])
    plot_images_in_grid(images, times, (sx, sy), zoom_size=zoom)


# Batch vetting of all 2+ telescope coincidences

Vet every coincident candidate in one pass and rank them. This mirrors the pipeline's own astrophysical-vs-artifact logic:

- **Tier** — present-telescope detection times agreeing to ≤0.2 s (`Tier3` for 3, `Tier2` for 2) is the astrophysical signature; a looser spread is `Tier1-like` (satellite/cosmic-ray-like).
- **Per-dip metrics** — matched-filter `significance`, fractional depth, and dip duration in frames (a 1-frame dip is cosmic-ray-like).
- **Missing telescopes** — flagged `OBSERVABLE` (was observing + star in FOV → should show a dip; reconstruct to confirm), `not observing`, `out of FOV`, or `no WCS`.

Each candidate gets a **verdict**: `LIKELY`, `NEEDS RECON` (an observable telescope is missing — reconstruct it), `REVIEW`, or `ARTIFACT? (loose timing)`. The pass is cheap (no WCS solves, no frame reads); it sets `detect_idx` to the first candidate worth drilling into.

In [ ]:
# ==================================================================
# Batch vetting of EVERY 2+ telescope coincident candidate.
# ------------------------------------------------------------------
# Replicates the pipeline's tier logic (tight time spread <=0.2s across 2-3
# telescopes at one RA/Dec = astrophysical; loose = artifact) and adds per-dip
# metrics.  Cheap only: no WCS solves, no frame IO -- it just FLAGS which
# candidates need a raw-frame reconstruction (done per-candidate in drill-in).
# ==================================================================
SIG_THRESHOLD  = 5.0   # matched-filter significance for a "real" dip
TIER_TIME_TOL  = 0.2   # s: tight-coincidence tolerance (simultaneous_occults.py)


def _to_float(s):
    try:
        return float(s)
    except (TypeError, ValueError):
        return np.nan


def dip_metrics(tel, fname):
    """Per-detection dip stats: significance, fractional depth, duration (frames)."""
    df, meta = load_lightcurve(tel, fname)
    flux = df["flux"].to_numpy(dtype=float)
    raw_mean = _to_float(meta.get("Raw lightcurve mean"))
    raw_std  = _to_float(meta.get("Raw lightcurve std"))
    if np.isnan(raw_mean):
        raw_mean = float(np.mean(flux))
    if np.isnan(raw_std):
        raw_std = float(np.std(flux))
    frac_depth = (raw_mean - flux.min()) / raw_mean if raw_mean else np.nan
    # Longest run of frames below mean - 3 sigma (1-frame dip => cosmic-ray-like).
    below = flux < (raw_mean - 3 * raw_std)
    dur = cur = 0
    for b in below:
        cur = cur + 1 if b else 0
        dur = max(dur, cur)
    return _to_float(meta.get("significance")), frac_depth, int(dur)


def _find_minute_cached(tel, event_dt, cache):
    for start, name, full in cache[tel.name]:
        if start <= event_dt < start + timedelta(minutes=1.1):
            return name, full
    return None


def vet_candidate(idx, cache):
    cand = candidates[idx]
    present, missing, radec = cand["present"], cand["missing"], cand["radec"]

    sigs, depths, durs, dip_times = [], [], [], []
    for name in present:
        sig, depth, dur = dip_metrics(TEL_BY_NAME[name], cand["files"][name])
        sigs.append(sig); depths.append(depth); durs.append(dur)
        dip_times.append(get_event_datetime(cand["files"][name]))

    spread = (max(dip_times) - min(dip_times)).total_seconds() if len(dip_times) > 1 else 0.0
    n = len(present)
    if spread <= TIER_TIME_TOL and n >= 3:
        tier = "Tier3"
    elif spread <= TIER_TIME_TOL and n == 2:
        tier = "Tier2"
    else:
        tier = "Tier1-like"

    # Why is each missing telescope missing? (cheap: dir listing + FOV check, no solve)
    miss_status, any_observable = {}, False
    for name in missing:
        tel = TEL_BY_NAME[name]
        md = _find_minute_cached(tel, cand["event_dt"], cache)
        if md is None:
            miss_status[name] = "not observing"
            continue
        in_fov, _ = check_in_fov(tel, md[0], radec)
        if in_fov is True:
            miss_status[name] = "OBSERVABLE"; any_observable = True
        elif in_fov is False:
            miss_status[name] = "out of FOV"
        else:
            miss_status[name] = "no WCS"

    min_sig = np.nanmin(sigs) if sigs else np.nan
    min_dur = min(durs) if durs else 0

    if tier == "Tier1-like":
        verdict = "ARTIFACT? (loose timing)"
    elif any_observable:
        verdict = "NEEDS RECON"
    elif (not np.isnan(min_sig) and min_sig >= SIG_THRESHOLD) and min_dur > 1:
        verdict = "LIKELY"
    else:
        verdict = "REVIEW"

    return {
        "idx": idx,
        "event": cand["event_dt"].strftime("%H:%M:%S.%f")[:-3],
        "RA": round(radec[0], 4), "Dec": round(radec[1], 4),
        "present": ",".join(present), "missing": ",".join(missing) or "-",
        "tier": tier, "dt_spread_s": round(spread, 3),
        "min_sig": round(float(min_sig), 2) if not np.isnan(min_sig) else np.nan,
        "min_dur": min_dur,
        "missing_status": ";".join(f"{k}:{v}" for k, v in miss_status.items()) or "-",
        "verdict": verdict,
    }


def vet_all():
    cache = {t.name: list_minute_dirs(t) for t in TELESCOPES}   # one dir listing per telescope
    df = pd.DataFrame([vet_candidate(i, cache) for i in range(len(candidates))])
    if len(df):
        rank = {"NEEDS RECON": 0, "LIKELY": 1, "REVIEW": 2, "ARTIFACT? (loose timing)": 3}
        df = df.sort_values("verdict", key=lambda s: s.map(rank), kind="stable").reset_index(drop=True)
    return df


vetting_df = vet_all()
print(vetting_df.to_string(index=False) if len(vetting_df) else "No 2+ telescope candidates.")
if len(vetting_df):
    print("\nVerdict counts:")
    print(vetting_df["verdict"].value_counts().to_string())

# Choose a candidate to drill into: first NEEDS RECON, else first LIKELY, else 0.
if len(vetting_df):
    needs = vetting_df[vetting_df.verdict == "NEEDS RECON"]
    likely = vetting_df[vetting_df.verdict == "LIKELY"]
    detect_idx = int((needs if len(needs) else likely if len(likely) else vetting_df).iloc[0]["idx"])
else:
    detect_idx = 0
print(f"\nDrill-in candidate: detect_idx = {detect_idx}")

# Drill into a candidate

`detect_idx` is set by the batch pass above (first `NEEDS RECON`, else first `LIKELY`). Change it to any row's `idx` to inspect a different candidate: per-telescope status + timing, the overlaid light curves, an on-demand reconstruction of a missing telescope, and the event-image grid.

In [ ]:
# Per-telescope status + light curves + first frames for the chosen candidate.
verify_candidate(detect_idx)
plot_candidate(detect_idx)
show_first_frames(detect_idx)

In [ ]:
# Reconstruct any missing-but-observable telescope (cross-boundary + on-demand WCS).
cand = candidates[detect_idx]
if cand["missing"]:
    overlay_reconstruction(detect_idx, cand["missing"][0])
else:
    print("All telescopes present for this candidate; nothing to reconstruct.")

In [ ]:
# Frames around the event for one present telescope (visual artifact check).
show_event_images(detect_idx, candidates[detect_idx]["present"][0])